# 03 — Physics-Aware PORTEX-Weighted GLM

Fits the interpretable PORTEX weighted logistic model with L1 penalty under LOSO CV. Coefficient analysis confirms Vulnerability (+17.61) and Hazard (+3.21) as primary risk drivers.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.models.portex_glm import PORTEXWeightedGLM
from src.evaluation.loso_cv import run_loso_cv
from src.data.preprocessing import FEATURE_COLS, GROUP_COL, LABEL_COL
from src.utils.visualization import plot_feature_importance

df = pd.read_parquet("../data/processed/portex_labeled.parquet")


In [ ]:
# ── Fit PORTEX-GLM under LOSO ───────────────────────────
model = PORTEXWeightedGLM(penalty="l1", C=1.0)
result = run_loso_cv(model, df, feature_cols=FEATURE_COLS,
                     label_col=LABEL_COL, group_col=GROUP_COL)
print(f"\nMean AUC: {result['mean_auc']:.4f} ± {result['std_auc']:.4f}")


In [ ]:
# ── Fit on full dataset for coefficient analysis ────────
X_full = df[FEATURE_COLS]
y_full = df[LABEL_COL]
model.fit(X_full, y_full)
model.print_summary()


In [ ]:
# ── Feature importance plot ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
plot_feature_importance(model.coefficients, ax=ax)
plt.savefig("../results/figures/03_portex_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Coefficient stability across LOSO folds ─────────────
coef_records = []
from sklearn.model_selection import LeaveOneGroupOut
logo = LeaveOneGroupOut()
groups = df[GROUP_COL].values
for train_idx, test_idx in logo.split(X_full, y_full, groups):
    m = PORTEXWeightedGLM(penalty="l1", C=1.0)
    m.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
    coef_records.append(m.coefficients)

coef_df = pd.DataFrame(coef_records)
print("Coefficient stability (mean ± std across LOSO folds):")
print(coef_df.agg(["mean","std"]).T)
